# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [3]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)


custom_profile = {
    "max_input_tokens": 128_000
}
model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    profile=custom_profile,
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

In [4]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户（老王）向AI打招呼并询问AI的身份，是一次简单的初次互动/破冰对话。目前没有明确的任务型目标。

## SUMMARY
- 用户自我介绍为"老王"。
- AI（系统设定为友好型AI助手）回应自己叫"小王"。
- 用户表示很高兴认识"小王"。
- 对话处于开场阶段，尚未涉及任何具体任务、决策或技术内容。
- 决定/结论：AI 在此次对话中被命名为"小王"。
- 被拒绝的选项：无。

## ARTIFACTS
None

## NEXT STEPS
- 等待用户提出具体的任务或问题。
- 保持友好助手的人设，用名"小王"继续与"老王"对话。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王别多心，我纯属抖个机灵。意思是：你现在觉得认识“小王”挺高兴，万一后面发现我有点贫嘴、爱开玩笑，可别嫌烦——所以说“高兴得太早”。

没有别的意思，要是这句让你不舒服了，我道个歉。你要是有正事，随时吩咐，我正经起来还是很靠谱的。


## 举例2：summary_prompt

In [6]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

对历史消息的摘要如下：

老王和助手进行了简单的自我介绍。老王先打招呼并询问助手身份，助手自称“小王”；随后老王表示很高兴认识助手。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，别误会，我是在开玩笑呢！

我是说，你上一句说“很高兴认识我”，而我作为助手，本来应该热切回应你的热情——结果我刚才那句回复写得有点“老司机”似的冷幽默，好像你在单方面高兴，而我要泼点冷水似的。其实不是那个意思啦，只是顺手调侃一下“高兴得太早”这个梗而已。

如果让你觉得不舒服了，那确实是我的不对。我重新来一遍：

**老王你好，很高兴认识你！以后有什么需要帮忙的，尽管说，我是认真来打工的小王。** 😄
